In [11]:
from rapidsegment import StrategicSegmentBuilder, StrategicSegmentScore, UniversalDataLoader


In [12]:
data = UniversalDataLoader(file_path = r"/workspaces/RapidSegment/student_burnout_dropout_dataset_2.csv").load()

In [13]:
import pyarrow as pa

# Dictionary comprehension to get counts for all columns
null_counts = {name: data[name].null_count for name in data.column_names}
print(null_counts)
# Output example: {'col1': 0, 'col2': 3, 'col3': 1}


{'Student_ID': 0, 'Age': 0, 'Gender': 0, 'Year_of_Study': 0, 'Department': 0, 'Residence_Type': 0, 'Attendance_Percent': 51, 'Study_Hours_Per_Day': 35, 'Previous_GPA': 35, 'Backlogs': 15, 'Sleep_Hours': 35, 'Screen_Time_Hours': 36, 'Exercise_Freq_Per_Week': 0, 'Social_Activity_Score': 40, 'Part_Time_Job': 0, 'Family_Income_Bracket': 0, 'Financial_Stress_Score': 43, 'Family_Support_Score': 44, 'Stress_Level': 49, 'Anxiety_Score': 39, 'Motivation_Score': 41, 'Peer_Pressure_Score': 41, 'Counseling_Access': 0, 'Burnout_Level': 0, 'Dropout_Risk': 0}


In [14]:

import pyarrow.compute as pc


# 2. Define your replacements
condition = pc.equal(data["Dropout_Risk"], "Yes")
new_values = pc.if_else (condition, 1, 0)

# 3. Replace the old column with the new computed array in the immutable table
new_table = data.set_column(
    data.schema.get_field_index("Dropout_Risk"), 
    "Dropout_Risk", 
    new_values
)


In [15]:
grid_config = {
    "min_sample_size": [10, 50],
    "min_lift": [2.0, 1.2]
}


In [16]:
builder = StrategicSegmentBuilder(
    target="Dropout_Risk",
    min_events=1,
    top_n_vars=10,
    max_segments=10,
    max_feature_reuse=1,
    param_grid=grid_config,
    enable_diversity=True,
    feature_groups=None,
    ignore_features=["Student_ID"]
)

In [17]:
segments_summary = builder.extract_segments(new_table)

2026-07-15 15:44:04,483 | INFO     | [builder.py:349] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-15 15:44:04,495 | INFO     | [builder.py:364] | Dynamic Grid Search Enabled: 4 total configurations.
2026-07-15 15:44:04,496 | INFO     | [builder.py:385] | Iteration 1 | Remaining Volume: 800 | Base Rate: 54.62%
2026-07-15 15:44:05,644 | INFO     | [builder.py:522] | Feature Usage Tracker Update -> 'Burnout_Level' used count = 1
2026-07-15 15:44:05,645 | INFO     | [builder.py:522] | Feature Usage Tracker Update -> 'Family_Income_Bracket' used count = 1
2026-07-15 15:44:05,645 | INFO     | [builder.py:522] | Feature Usage Tracker Update -> 'Study_Hours_Per_Day' used count = 1
2026-07-15 15:44:05,646 | INFO     | [builder.py:537] | Segment 1 Captured (Size Floor: 50 | Lift Floor: 1.2): Burnout_Level IN ('High') AND Family_Income_Bracket IN ('Lower-Middle') AND (Study_Hours_Per_Day >= 0.05 AND Study_Hours_Per_Day < 3.45)
2026-07-15 15:44:05,658 | INFO     | [builder.py:385] | 

In [18]:
import pandas as pd
segments_df = pd.DataFrame(segments_summary)
print("\nExtracted Segment Profiles:")
segments_df[["segment_id", "count", "lift", "meta_applied_sample_size", "sql_filter"]]



Extracted Segment Profiles:


,segment_id,count,lift,meta_applied_sample_size,sql_filter
0,1,51,1.758873,50,Burnout_Level IN ('High') AND Family_Income_Br...
1,2,90,1.333161,50,Attendance_Percent < 52.45
2,3,93,1.286440,50,Stress_Level >= 7.45
3,4,10,1.935918,10,(Peer_Pressure_Score >= 3.65 AND Peer_Pressure...
4,5,74,1.208411,50,Exercise_Freq_Per_Week < 0.50
5,6,119,1.216420,50,(Year_of_Study >= 1.50 AND Year_of_Study < 2.50)
6,7,13,1.895821,10,(Previous_GPA >= 7.62 AND Previous_GPA < 7.90)...
7,8,16,1.461496,10,Anxiety_Score >= 8.35
8,9,66,1.229128,50,Part_Time_Job IN ('Yes')
9,10,55,1.223797,50,Age < 19.50


In [19]:
coverage_report = builder.evaluate_final_coverage(new_table)
print("\nCascading Portfolio Coverage Analysis:")
pd.DataFrame(coverage_report)



Cascading Portfolio Coverage Analysis:


,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,173,57.0,32.947977,54.625,21.625,0.603167
1,1,51,49.0,96.078431,54.625,6.375,1.758873
2,2,39,33.0,84.615385,54.625,4.875,1.549023
3,3,99,64.0,64.646465,54.625,12.375,1.183459
4,4,10,9.0,90.000000,54.625,1.250,1.647597
5,5,89,53.0,59.550562,54.625,11.125,1.090170
6,6,145,76.0,52.413793,54.625,18.125,0.959520
7,7,14,10.0,71.428571,54.625,1.750,1.307617
8,8,21,13.0,61.904762,54.625,2.625,1.133268
9,9,87,40.0,45.977011,54.625,10.875,0.841684


In [23]:
import duckdb
con = duckdb.connect(database='temp_scoring.db', read_only=False)

In [24]:
con.execute("CREATE TABLE IF NOT EXISTS scored_data AS SELECT Student_ID, Dropout_Risk FROM new_table")

In [25]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   Burnout_Level=<ArrowStringArray>
['High']
Length: 1, dtype: str & Family_Income_Bracket=<ArrowStringArray>
['Lower-Middle']
Length: 1, dtype: str & Study_Hours_Per_Day=[0.05, 3.45)
SQL Filter: Burnout_Level IN ('High') AND Family_Income_Bracket IN ('Lower-Middle') AND (Study_Hours_Per_Day >= 0.05 AND Study_Hours_Per_Day < 3.45)
--------------------------------------------------
Segment ID: 2
Raw Rule:   Attendance_Percent=(-inf, 52.45)
SQL Filter: Attendance_Percent < 52.45
--------------------------------------------------
Segment ID: 3
Raw Rule:   Stress_Level=[7.45, inf)
SQL Filter: Stress_Level >= 7.45
--------------------------------------------------
Segment ID: 4
Raw Rule:   Peer_Pressure_Score=[3.65, 4.05) & Screen_Time_Hours=[6.15, 7.25)
SQL Filter: (Peer_Pressure_Score >= 3.65 AND Peer_Pressure_Score < 4.05) AND (Screen_Time_Hours >= 6.15 AND Screen_Time_Hours < 7.25)
--------------------------------------------------
Segm

In [26]:
con.close()